In [1]:
!pip install --upgrade accelerate
!pip install --upgrade bitsandbytes
!pip install --upgrade peft
!pip install --upgrade sentencepiece
!pip install --upgrade transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.6/92.6 MB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.7/174.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 7.0 MB/s eta 0:00:00


In [2]:
import csv
import gc
import re

from bs4 import BeautifulSoup
import pandas as pd
from peft import PeftModel
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, BertTokenizer, LlamaForCausalLM, LlamaTokenizerFast, pipeline
import torch
from tqdm import tqdm
from google.colab import drive

In [7]:
test_text = "ASML Holding's Q3 Earnings Preview\n\n ASML Holding  (NASDAQ: ASML ) announces its next round of earnings this Wednesday. Here is Benzinga's everything-that-matters guide for the\xa0Q3 earnings announcement. \n Earnings and Revenue \n ASML Holding EPS is expected to be around $1.82, according to sell-side analysts. Sales will likely be near $3.21 billion. \n In the same quarter last year, ASML Holding reported EPS of $1.52 on revenue of $2.89 billion. If the company were to match the consensus estimate when it reports Wednesday, EPS would be up 19.74 percent. Revenue would be up 11.15 percent from the same quarter last year. Here's how the company's reported EPS has compared to analyst estimates in the past: \n \xa0 \n \n \n \n Quarter \n Q2 2018 \n Q1 2018 \n Q4 2017 \n Q3 2017 \n \n \n EPS Estimate \n 1.29 \n 1.13 \n 1.29 \n 1.29 \n \n \n EPS Actual \n 1.59 \n 1.56 \n 1.82 \n 1.52 \n \n \n \n Stock Performance \n Over the past 52-week period, shares of ASML Holding have declined 1.67 percent. Analysts\xa0have adjusted their estimates higher for EPS and revenues over the past 90 days. Analysts generally rate ASML Holding stock as Neutral. The strength of this rating has maintained conviction over the past three months. \n Conference Call \n ASML Holding's Q3 conference call is scheduled to begin at 8:00 a.m. ET and can be accessed here:   \n"

In [63]:
pipe = pipeline("text-classification", model="yiyanghkust/finbert-tone")
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-tone')
pipe(test_text)

[{'label': 'Neutral', 'score': 0.5125790238380432}]

In [62]:
# Another LLM worth trying but not as good
pipe_2 = pipeline("text-classification", model="mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis")
pipe_2(test_text)
# tokenizer = AutoTokenizer.from_pretrained("mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis")

config.json:   0%|          | 0.00/933 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[{'label': 'positive', 'score': 0.9996575117111206}]

In [3]:
df_news = pd.read_csv('ASML_news.csv')
df_news.head()

,text,datetime
0,18 Stocks Moving In Monday's Pre-Market Sessio...,2016-12-05 08:29:19-05:00
1,Bank of America Upgrades ASML Holding N.V. to ...,2016-12-19 06:32:08-05:00
2,"Benzinga's Top Upgrades, Downgrades For Januar...",2017-01-17 09:22:35-05:00
3,"Earnings Scheduled For January 18, 2017\n\n \r...",2017-01-18 04:33:08-05:00
4,ASML +4.3% Premarket @$120.91; CFO Says 2017 i...,2017-01-18 07:55:31-05:00


In [9]:
%%time
df_news['token_len'] = df_news['text'].apply(lambda text: len(tokenizer.tokenize(text)))
df_news.head()

CPU times: user 14.8 s, sys: 9.88 ms, total: 14.8 s
Wall time: 14.9 s


,text,datetime,token_len
0,18 Stocks Moving In Monday's Pre-Market Sessio...,2016-12-05 08:29:19-05:00,803
1,Bank of America Upgrades ASML Holding N.V. to ...,2016-12-19 06:32:08-05:00,13
2,"Benzinga's Top Upgrades, Downgrades For Januar...",2017-01-17 09:22:35-05:00,1064
3,"Earnings Scheduled For January 18, 2017\n\n \r...",2017-01-18 04:33:08-05:00,737
4,ASML +4.3% Premarket @$120.91; CFO Says 2017 i...,2017-01-18 07:55:31-05:00,24


In [17]:
print(f"Percentage of texts with token length greater than 450: {df_news['token_len'].gt(450).sum()/len(df_news):.2%}")
df_news['token_len'].describe()

Percentage of texts with token length greater than 450: 53.82%


count      680.000000
mean      1491.572059
std       3261.417956
min          7.000000
25%        119.500000
50%        475.500000
75%        993.000000
max      29717.000000
Name: token_len, dtype: float64

In [18]:
display(df_news.loc[df_news['token_len'].idxmin()]['text'])

'UBS Upgrades ASML Holding to Buy\n\n'

In [19]:
display(df_news.loc[df_news['token_len'].idxmax()]['text'])

"Stocks That Hit 52-Week Lows On Monday. \xa0 \n During Monday's trading, 1214 companies set new 52-week lows. \n Facts of Interest About Today's 52-Week Lows: \n \n The largest company by market cap to set a new 52-week low was  Microsoft (NASDAQ: MSFT ) . \n Quoin Pharmaceuticals (NASDAQ: QNRX )  was the smallest company by market cap to set a new 52-week low. \n DexCom (NASDAQ: DXCM ) 's stock fell the most, as it traded down 76.5% to reach a new 52-week low. \n La Jolla Pharmaceutical (NASDAQ: LJPC ) 's stock made the biggest bounce back, actually moving up 0.0% shortly after hitting a new 52-week low. \n \n On Monday, the following stocks broke to new 52-week lows: \n \n Bank of America (NYSE: BAC )  stock hit a yearly low of $31.99. The stock was down 3.01% for the day. \n ASML Holding (NASDAQ: ASML )  shares moved down 4.86% on Monday to hit a new 52-week low of $491.62, drifting down 4.86%. \n Walt Disney (NYSE: DIS )  stock broke to a new 52-week low of $94.83 on Monday. Share

In [61]:
print(df_news['text'][4])
pipe(df_news['text'][4])

ASML +4.3% Premarket @$120.91; CFO Says 2017 is Going to be a Great Year




[{'label': 'Positive', 'score': 1.0}]

In [55]:
def convert_to_score(sentiment):
    if 'negative' in sentiment.lower():
        return -1
    elif 'positive' in sentiment.lower():
        return 1
    else:
        return 0

def sliding_window_chunks(text, window_size, stride):
    tokens = tokenizer(text)["input_ids"]
    token_chunks = [tokens[i:i + window_size] for i in range(0, len(tokens), stride) if i + window_size <= len(tokens)]
    return token_chunks

# parameters for chunking
window_size = 450
stride = 225  # overlap length

In [57]:
sentiment_scores = []
for index, row in tqdm(df_news.iterrows(), total=len(df_news), desc="Processing news texts..."):
    text = row['text']
    if len(tokenizer(text)["input_ids"]) > window_size:
        scores = []
        for chunk in sliding_window_chunks(text, window_size, stride):
            scores.append(convert_to_score(pipe(tokenizer.decode(chunk))[0]['label']))
        sentiment_score = max(set(scores), key=scores.count)
        sentiment_scores.append(sentiment_score)
    else:
        sentiment_scores.append(convert_to_score(pipe(text)[0]['label']))

Processing news texts...: 100%|██████████| 680/680 [1:34:33<00:00,  8.34s/it]


In [58]:
with open('sentiment_scores.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows([sentiment_scores])

## FinGPT v3
Another LLM worth trying, but not very scalable and takes too much GPU RAM for inferencing

In [ ]:
base_model = "NousResearch/Llama-2-13b-hf"
peft_model = "FinGPT/fingpt-sentiment_llama2-13b_lora"
tokenizer = LlamaTokenizerFast.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token
model = LlamaForCausalLM.from_pretrained(base_model, device_map='cuda:0', load_in_8bit=True) # GPU is used
model = PeftModel.from_pretrained(model, peft_model)
model = model.eval()

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/33.4k [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/175 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:381: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:386: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(


adapter_config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

adapter_model.bin:   0%|          | 0.00/19.7M [00:00<?, ?B/s]

In [ ]:
# prompt = [
# '''Instruction: What is the sentiment of this news? Choose an answer from {negative/neutral/positive}
# Input: FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is aggressively pursuing its growth strategy by increasingly focusing on technologically more demanding HDI printed circuit boards PCBs .
# Answer: ''',
# '''Instruction: What is the sentiment of this news? Choose an answer from {negative/neutral/positive}
# Input: According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .
# Answer: ''',
# '''Instruction: What is the sentiment of this news? Choose an answer from {negative/neutral/positive}
# Input: A tinyurl link takes users to a scamming site promising that users can earn thousands of dollars by becoming a Google ( NASDAQ : GOOG ) Cash advertiser .
# Answer: ''',
# ]

# with torch.inference_mode():
#     tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512)
#     res = model.generate(**tokens, max_length=512)
#     res_sentences = [tokenizer.decode(i) for i in res]
#     out_text = [o.split("Answer: ")[1] for o in res_sentences]

# for sentiment in out_text:
#     print(sentiment)

# del res, tokens, res_sentences, out_text
# gc.collect()
# torch.cuda.empty_cache()

In [ ]:
symbol = 'ASML'
sentiment_scores = []

with torch.inference_mode():
    for index, row in tqdm(df_news.iterrows(), total=len(df_news), desc="Processing news texts..."):
        text = row['text']
        # if text is about to exceed maximum input length
        if len(tokenizer(text)["input_ids"]) > window_size:
            # the following commented section is for inferencing in batches if GPU has **enough** memory

            # prompts = [f'''Instruction: What is the sentiment of {symbol} in this news?
            # Choose an answer from {{negative/neutral/positive}}
            # Input: {tokenizer.decode(chunk)}
            # Answer: ''' for chunk in sliding_window_chunks(text, window_size, stride)]
            # tokens = tokenizer(prompts, return_tensors='pt', padding=True, max_length=512, truncation=True)
            # # tokens = {k: v.to("cuda:0") for k, v in tokens.items()} # move tokens to the same device as the model
            # res = model.generate(**tokens, max_length=512)
            # res_sentences = [tokenizer.decode(i) for i in res]
            # sentiments = [o.split("Answer: ")[1] for o in res_sentences]
            # scores = [convert_to_score(sentiment) for sentiment in sentiments]
            # # free memory
            # del tokens, res, res_sentences
            # gc.collect()
            # torch.cuda.empty_cache()
            # # aggregate scores by majority voting
            # sentiment_scores.append(max(set(scores), key=scores.count))

            scores = []
            for chunk in sliding_window_chunks(text, window_size, stride):
                prompt = f'''Instruction: What is the sentiment of {symbol} in this news?
                            Choose an answer from {{negative/neutral/positive}}
                            Input: {tokenizer.decode(chunk)}
                            Answer: '''
                tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512, truncation=True)
                # tokens = {k: v.to("cuda:0") for k, v in tokens.items()}
                res = model.generate(**tokens, max_length=512)
                res_sentence = [tokenizer.decode(i) for i in res]
                sentiment = [o.split("Answer: ")[1] for o in res_sentence][0]
                scores.append(convert_to_score(sentiment))
                # free memory
                del tokens, res, res_sentence
                gc.collect()
                torch.cuda.empty_cache()
            # aggregate scores by majority voting
            sentiment_scores.append(max(set(scores), key=scores.count))
            print([index, max(set(scores), key=scores.count)])
        else:
            prompt = [
            f'''Instruction: What is the sentiment of {symbol} in this news?
            Choose an answer from {{negative/neutral/positive}}
            Input: {text}
            Answer: '''
            ]
            tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512, truncation=True)
            # tokens = {k: v.to("cuda:0") for k, v in tokens.items()} # move tokens to the same device as the model
            response = model.generate(**tokens, max_length=512)
            response_sentences = [tokenizer.decode(i) for i in response]
            sentiment = [o.split("Answer: ")[1] for o in response_sentences][0]
            sentiment_scores.append(convert_to_score(sentiment))
            print([index, convert_to_score(sentiment)])
            # free memory
            del tokens, response, response_sentences
            gc.collect()
            torch.cuda.empty_cache()